In [0]:
# Load Silver tables
silver_orders = spark.table(
    "dbacademy.retailflow.silver_orders"
)

silver_order_items = spark.table(
    "dbacademy.retailflow.silver_order_items"
)

silver_products = spark.table(
    "dbacademy.retailflow.silver_products"
)

silver_stores = spark.table(
    "dbacademy.retailflow.silver_stores"
)

# Load Gold tables
gold_daily_sales = spark.table(
    "dbacademy.retailflow.gold_daily_sales"
)

gold_product_performance = spark.table(
    "dbacademy.retailflow.gold_product_performance"
)

gold_store_performance = spark.table(
    "dbacademy.retailflow.gold_store_performance"
)

gold_monthly_sales = spark.table(
    "dbacademy.retailflow.gold_monthly_sales"
)

print("All Silver and Gold tables loaded successfully!")

All Silver and Gold tables loaded successfully!


In [0]:
from pyspark.sql.functions import col

quality_results = []

def run_check(check_name, failed_records):
    failed_count = failed_records.count()

    status = "PASS" if failed_count == 0 else "FAIL"

    quality_results.append({
        "check_name": check_name,
        "failed_records": failed_count,
        "status": status
    })

# Orders checks
run_check(
    "Duplicate Order IDs",
    silver_orders.groupBy("order_id")
    .count()
    .filter(col("count") > 1)
)

run_check(
    "Missing Customer IDs",
    silver_orders.filter(col("customer_id").isNull())
)

# Order items checks
run_check(
    "Duplicate Order Item IDs",
    silver_order_items.groupBy("order_item_id")
    .count()
    .filter(col("count") > 1)
)

run_check(
    "Invalid Quantities",
    silver_order_items.filter(col("quantity") <= 0)
)

# Product checks
run_check(
    "Missing Product Categories",
    silver_products.filter(col("category").isNull())
)

run_check(
    "Invalid Product Prices",
    silver_products.filter(
        col("price").isNull() | (col("price") <= 0)
    )
)

# Store checks
run_check(
    "Duplicate Store IDs",
    silver_stores.groupBy("store_id")
    .count()
    .filter(col("count") > 1)
)

run_check(
    "Missing Store Details",
    silver_stores.filter(
        col("store_name").isNull() | col("city").isNull()
    )
)

# Gold checks
run_check(
    "Negative Daily Revenue",
    gold_daily_sales.filter(col("total_revenue") < 0)
)

run_check(
    "Negative Product Revenue",
    gold_product_performance.filter(col("total_revenue") < 0)
)

run_check(
    "Negative Store Revenue",
    gold_store_performance.filter(col("total_revenue") < 0)
)

print("All data quality checks executed!")

All data quality checks executed!


In [0]:
quality_report = spark.createDataFrame(quality_results)

display(
    quality_report.orderBy("status", "check_name")
)

check_name,failed_records,status
Duplicate Order IDs,0,PASS
Duplicate Order Item IDs,0,PASS
Duplicate Store IDs,0,PASS
Invalid Product Prices,0,PASS
Invalid Quantities,0,PASS
Missing Customer IDs,0,PASS
Missing Product Categories,0,PASS
Missing Store Details,0,PASS
Negative Daily Revenue,0,PASS
Negative Product Revenue,0,PASS
